In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [3]:
df = pd.read_csv('/content/wows_ship_stats.csv')

In [4]:
df.head()

,Ship,Tier,Class,Nation,Players,Battles,Base XP,Damage,Frags,Win rate,Capture,Defence,Spotting,Potential,Aircraft,Survival rate
0,Patrie,11,BB,France,9092,337170,1388,125865,0.936,0.512,2.13,4.74,19358,1809099,5.60,0.426
1,Hannover,11,BB,Germany,16503,319701,1150,86471,0.633,0.485,1.81,4.16,20080,1930510,6.42,0.240
2,Satsuma,11,BB,Japan,40417,1562549,1246,112667,0.809,0.499,1.32,3.21,17402,1730502,4.15,0.469
3,Devastation,11,BB,U.K.,5660,140172,1323,127807,0.766,0.499,1.98,4.78,20776,1896948,4.71,0.499
4,Maine,11,BB,U.S.A.,10744,352970,1344,114812,0.890,0.513,2.32,4.11,22194,2078203,7.70,0.445


In [5]:
x = df.drop(['Class','Nation','Tier','Ship','Survival rate'],axis = 1)
y = df.iloc[:,-1]

In [6]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.2,random_state = 22)

In [7]:
x_train.head()

,Players,Battles,Base XP,Damage,Frags,Win rate,Capture,Defence,Spotting,Potential,Aircraft
915,101445,718314,565,26809,1.730,0.540,9.56,8.71,2857,213519,0.00
342,62054,600348,670,21349,0.543,0.508,12.80,1.12,11259,103415,0.00
486,79122,6637981,961,39206,0.578,0.510,7.23,5.54,20580,605823,2.08
792,87575,5184277,1068,34631,0.675,0.503,29.32,8.36,24156,504426,3.79
891,51864,808893,708,18174,0.572,0.490,18.35,5.72,11746,360949,0.70


In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [9]:
rt = DecisionTreeRegressor(
    criterion='squared_error'

)
rt.fit(x_train,y_train)

DecisionTreeRegressor()

In [10]:
y_pred = rt.predict(x_test)

In [11]:
r2_score(y_test,y_pred)

0.5586571592757479

# Hyperparameter Tuning

In [36]:
param_grid = {
    'max_depth':[2,3,5,10,None],
    'criterion':['squared_error','absolute_error','poisson'],
    'max_features' : [0.25,0.5,1.5,3,5],
    'min_samples_split' : [0.25,0.5,1.0]
}

In [37]:
reg = GridSearchCV(DecisionTreeRegressor(),param_grid= param_grid)

In [38]:
reg.fit(x_train,y_train)

GridSearchCV(estimator=DecisionTreeRegressor(),
             param_grid={'criterion': ['squared_error', 'absolute_error',
                                       'poisson'],
                         'max_depth': [2, 3, 5, 10, None],
                         'max_features': [0.25, 0.5, 1.5, 3, 5],
                         'min_samples_split': [0.25, 0.5, 1.0]})

In [39]:
reg.best_score_

np.float64(0.62902600304235)

In [40]:
reg.best_params_

{'criterion': 'absolute_error',
 'max_depth': None,
 'max_features': 0.5,
 'min_samples_split': 0.25}

## Feature Importnace

In [30]:
feature_names = x.columns

In [31]:
for importance, name in sorted(zip(rt.feature_importances_, feature_names), reverse=True):
    print(name, importance)

Capture 0.5752014923733794
Base XP 0.09535882641285155
Win rate 0.07900558659344781
Frags 0.05578637214075178
Damage 0.04583835357676958
Spotting 0.04402266501575586
Potential 0.04209793586973095
Aircraft 0.02398410054369968
Defence 0.020967313999630997
Players 0.011669465199691973
Battles 0.006067888274290462
